# Notebook 6 — Feature Engineering

**Goals**

1. Build the four families of features that almost every supervised
   forecasting model needs:
   - **Calendar / date features** (year, month, day-of-week, holidays, …)
   - **Lag features** (target shifted by 1, 7, 14, … periods)
   - **Rolling-window features** (mean / std / min / max over recent
     periods)
   - **Static & dynamic regressors** encoded properly
2. Apply *numeric transformations* — log, Box-Cox, Yeo-Johnson — that
   stabilise variance.
3. Scale numeric features for distance-based / linear models.
4. Encode categorical columns with the right method for the situation.

>  **Leakage warning.** Several of these techniques are *easy to do
> wrong*. We highlight every place where leakage can creep in and how to
> avoid it.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


In [4]:
import forecasting_toolkit as ft

# Use a cleaned dataframe — for tutorial flow we just re-run the
# minimum cleaning pipeline from Notebook 5 here. In your own work
# you'd pass the cleaned dataframe in directly.
df_clean = ft.preprocessing.fill_time_gaps(df, spec, fill_value=None, add_indicator=True)
if STATIC_COLS:
    df_clean = ft.preprocessing.ffill_bfill_static(df_clean, spec)
df_clean = ft.preprocessing.impute_missing(df_clean, spec, cols=TARGET_COL, method='zero')
print(f'Cleaned shape: {df_clean.shape}')


Cleaned shape: (5025926, 39)


## 6.1 Calendar / date features

`add_date_features` pulls a standard set of date attributes from the
spec's date column. With `cyclical=True` (default) it also encodes
**month**, **day-of-week**, and **day-of-year** as sin/cos pairs so models
see December and January as adjacent points on a circle, not as 12 and 1.

### Why cyclical encoding?

If a model sees `month` as a plain integer, December (12) and January
(1) look 11 apart — but they should be 1 apart. Encoding as
`sin(2π · month / 12)` and `cos(2π · month / 12)` puts both close
together in 2-D space.


In [5]:
df_feat = ft.features.add_date_features(df_clean, spec,
                                        features=['year', 'month', 'dayofweek',
                                                  'dayofyear', 'is_weekend',
                                                  'is_month_end', 'is_quarter_end'],
                                        cyclical=True)
new_cols = [c for c in df_feat.columns if c not in df_clean.columns]
print('Added columns:', new_cols)
df_feat[[DATE_COL] + new_cols].head()


Added columns: ['dayofweek', 'dayofyear', 'is_weekend', 'is_quarter_end', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos']


,date,dayofweek,dayofyear,is_weekend,is_quarter_end,month_sin,month_cos,dayofweek_sin,dayofweek_cos,dayofyear_sin,dayofyear_cos
0,2022-07-18,0,199,0,0,-0.5,-0.866025,0.000000,1.000000,-0.280231,-0.959933
1,2022-07-19,1,200,0,0,-0.5,-0.866025,0.781831,0.623490,-0.296713,-0.954967
2,2022-07-20,2,201,0,0,-0.5,-0.866025,0.974928,-0.222521,-0.313107,-0.949718
3,2022-07-21,3,202,0,0,-0.5,-0.866025,0.433884,-0.900969,-0.329408,-0.944188
4,2022-07-22,4,203,0,0,-0.5,-0.866025,-0.433884,-0.900969,-0.345612,-0.938377


### Visualise the cyclical encoding


In [6]:
import numpy as np
import plotly.graph_objects as go
fig = go.Figure()
months = np.arange(1, 13)
fig.add_trace(go.Scatter(
    x=np.sin(2 * np.pi * months / 12),
    y=np.cos(2 * np.pi * months / 12),
    text=[f'M{m}' for m in months], mode='markers+text',
    textposition='top center',
    marker=dict(size=14, color='#2563EB')))
fig.update_layout(
    title='Cyclical (sin, cos) encoding of month — Dec and Jan are neighbours',
    xaxis_title='month_sin', yaxis_title='month_cos',
    template='plotly_white', height=440,
    xaxis=dict(scaleanchor='y', scaleratio=1))
fig.show()


## 6.2 Lag features

A lag feature is the target shifted by *k* periods **within each
forecast key**. They are the simplest way to give a non-temporal model
(XGBoost, LightGBM, linear regression) some memory of the past.

### Picking lags

| Frequency | Common lags                                  | Reasoning                                |
|-----------|----------------------------------------------|------------------------------------------|
| Daily     | 1, 7, 14, 28, 365                            | yesterday / weekly / monthly / annual    |
| Weekly    | 1, 2, 4, 13, 52                              | week-on-week / monthly / quarterly / yearly |
| Hourly    | 1, 24, 168                                   | hour / day / week                        |

> The first *k* rows of each series will be NaN where the lag walks off
> the edge. Drop them before training, or let your model handle NaNs.


In [9]:
df_feat = ft.features.add_lag_features(df_feat, spec, lags=[1, 7, 14, 28])
lag_cols = [c for c in df_feat.columns if c.startswith(f'{TARGET_COL}_lag_')]
print('Added lag columns:', lag_cols)
df_feat[KEY_COLS + [DATE_COL, TARGET_COL] + lag_cols].head(10)


Added lag columns: ['sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28']


,unique_id,date,sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28
0,0,2022-07-18,3.97,NaN,NaN,NaN,NaN
1,0,2022-07-19,73.36,3.97,NaN,NaN,NaN
2,0,2022-07-20,558.09,73.36,NaN,NaN,NaN
3,0,2022-07-21,14.03,558.09,NaN,NaN,NaN
4,0,2022-07-22,558.53,14.03,NaN,NaN,NaN
5,0,2022-07-23,453.38,558.53,NaN,NaN,NaN
6,0,2022-07-24,475.53,453.38,NaN,NaN,NaN
7,0,2022-07-25,607.79,475.53,3.97,NaN,NaN
8,0,2022-07-26,366.61,607.79,73.36,NaN,NaN
9,0,2022-07-27,117.72,366.61,558.09,NaN,NaN


## 6.3 Rolling-window features (with leakage guard)

Rolling stats summarise the recent past — a 7-day mean is a smoothed
short-term trend, a 28-day std measures recent volatility.

### The leakage trap

A naïve `df[col].rolling(7).mean()` at time *t* uses the value at *t*
**itself**. That's leakage if you're predicting that value!

The toolkit's `add_rolling_features` shifts the target by `shift=1`
before rolling, so the value at time *t* is computed from the window
`[t−7, t−1]` — never from *t* itself.


In [12]:
df_feat = ft.features.add_rolling_features(
    df_feat, spec,
    windows=[7, 14, 28],
    stats=['mean', 'std', 'min', 'max'],
    shift=1
)
roll_cols = [c for c in df_feat.columns if '_roll_' in c]
print(f'Added {len(roll_cols)} rolling columns')
print('Sample:', roll_cols[:6])
df_feat[KEY_COLS + [DATE_COL, TARGET_COL] + roll_cols[:4]].head(10)


Added 12 rolling columns
Sample: ['sales_roll_mean_7', 'sales_roll_std_7', 'sales_roll_min_7', 'sales_roll_max_7', 'sales_roll_mean_14', 'sales_roll_std_14']


,unique_id,date,sales,sales_roll_mean_7,sales_roll_std_7,sales_roll_min_7,sales_roll_max_7
0,0,2022-07-18,3.97,NaN,NaN,NaN,NaN
1,0,2022-07-19,73.36,3.970000,NaN,3.97,3.97
2,0,2022-07-20,558.09,38.665000,49.066140,3.97,73.36
3,0,2022-07-21,14.03,211.806667,301.890466,3.97,558.09
4,0,2022-07-22,558.53,162.362500,265.588914,3.97,558.09
5,0,2022-07-23,453.38,241.596000,290.332294,3.97,558.53
6,0,2022-07-24,475.53,276.893333,273.696334,3.97,558.53
7,0,2022-07-25,607.79,305.270000,260.885765,3.97,558.53
8,0,2022-07-26,366.61,391.530000,243.932861,14.03,607.79
9,0,2022-07-27,117.72,433.422857,201.710692,14.03,607.79


### Eyeball the smoothing on one series


In [13]:
import plotly.graph_objects as go

key = df_feat[KEY_COLS].drop_duplicates().iloc[0].to_dict()
sub = ft.data_io.get_series(df_feat, spec, key).sort_values(DATE_COL)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sub[DATE_COL], y=sub[TARGET_COL],
                         name='target', line=dict(color='#9CA3AF', width=1)))
fig.add_trace(go.Scatter(x=sub[DATE_COL], y=sub[f'{TARGET_COL}_roll_mean_7'],
                         name='roll_mean_7', line=dict(color='#2563EB', width=1.6)))
fig.add_trace(go.Scatter(x=sub[DATE_COL], y=sub[f'{TARGET_COL}_roll_mean_28'],
                         name='roll_mean_28', line=dict(color='#F59E0B', width=1.6)))
fig.update_layout(title=f'Rolling means — {key}',
                  template='plotly_white', height=420,
                  xaxis_title='Date', yaxis_title=TARGET_COL)
fig.show()


## 6.4 Numeric transformations

Many forecasting targets are **right-skewed** with a long tail. Linear
models, Gaussian-noise assumptions and even gradient-boosted ensembles
benefit when we transform the target to look more symmetric.

| Transform   | Formula                              | Notes                                                |
|-------------|--------------------------------------|------------------------------------------------------|
| `log(1+x)`  | `np.log1p(x)`                        | Cheapest. Requires `x ≥ 0`. Adds 1 so log(0) is OK. |
| **Box-Cox** | `(x^λ − 1) / λ` (or `log(x)` if λ=0) | Estimates λ from data. Requires `x > 0`.            |
| **Yeo-Johnson** | piecewise; works for negatives    | Best general-purpose choice when sign mixes.        |

> Always remember to **invert** the transform on predictions before
> reporting metrics in the original units.


In [14]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Sample the cleaned target for speed
sample = df_feat[TARGET_COL].dropna().sample(min(20_000, len(df_feat)), random_state=0)

log_x = ft.features.log_transform(sample, plus_one=True)
bc_x, bc_lam = ft.features.boxcox_transform(sample[sample > 0])
yj_x, yj_lam = ft.features.yj_transform(sample)

print(f'Box-Cox λ      : {bc_lam:.3f}')
print(f'Yeo-Johnson λ  : {yj_lam:.3f}')

fig = make_subplots(rows=1, cols=4, subplot_titles=['original', 'log(1+x)',
                                                    f'Box-Cox (λ={bc_lam:.2f})',
                                                    f'Yeo-Johnson (λ={yj_lam:.2f})'])
fig.add_trace(go.Histogram(x=sample,  marker_color='#9CA3AF', nbinsx=60), row=1, col=1)
fig.add_trace(go.Histogram(x=log_x,   marker_color='#2563EB', nbinsx=60), row=1, col=2)
fig.add_trace(go.Histogram(x=bc_x,    marker_color='#10B981', nbinsx=60), row=1, col=3)
fig.add_trace(go.Histogram(x=yj_x,    marker_color='#F59E0B', nbinsx=60), row=1, col=4)
fig.update_layout(height=380, template='plotly_white', showlegend=False,
                  title='Numeric transformations — distributions before/after')
fig.show()


Box-Cox λ      : -0.057
Yeo-Johnson λ  : 0.067


## 6.5 Scaling

Scaling matters for distance-based (k-NN), kernel (SVM) and gradient-
descent (neural nets, regularised linear) models. **Tree-based models
do not need it**, but it doesn't hurt them either.

| Scaler      | What it does                                  | When to prefer                  |
|-------------|------------------------------------------------|---------------------------------|
| `standard`  | (x − mean) / std                               | Default; assumes ~normal        |
| `minmax`    | (x − min) / (max − min) → [0, 1]               | Bounded inputs (neural nets)    |
| `robust`    | (x − median) / IQR                             | Heavy tails / lots of outliers  |

### The leakage trap

**Fit the scaler on the training split only.** Then apply the same
scaler to validation and test. If you fit on the whole dataset, the
scaler "sees" future data through the global mean / std — a subtle leak
that inflates validation accuracy.


In [15]:
# Demonstrate fit-on-train-then-apply pattern with a placeholder split.
# Notebook 7 covers proper time-based splits — use those in real work.
import numpy as np

target = df_feat[TARGET_COL].dropna().values
cut = int(0.8 * len(target))                # placeholder split for demo only
train_vals = target[:cut]
test_vals  = target[cut:]

scaler = ft.features.fit_scaler(train_vals, method='standard')
train_scaled = ft.features.apply_scaler(train_vals, scaler)
test_scaled  = ft.features.apply_scaler(test_vals,  scaler)

print(f'Train  | mean={np.mean(train_scaled):.3f}  std={np.std(train_scaled):.3f}')
print(f'Test   | mean={np.mean(test_scaled):.3f}  std={np.std(test_scaled):.3f}')
print('(Train mean ≈ 0, std ≈ 1 by construction. Test stats float around them.)')


Train  | mean=0.000  std=1.000
Test   | mean=0.181  std=3.095
(Train mean ≈ 0, std ≈ 1 by construction. Test stats float around them.)


## 6.6 Categorical encoding

Most forecasting datasets have categorical columns: store type, product
family, country, day-of-week as a string, etc. Models can't ingest
strings, so we encode them as numbers.

| Method            | Output cardinality | Strengths                                  | Cautions                                   |
|-------------------|-------------------|--------------------------------------------|--------------------------------------------|
| **Label**         | 1 column          | Simple. Fine for tree models.              | Imposes false ordering on linear models.   |
| **One-hot**       | K columns         | Safe for any model.                        | Explodes if K is huge.                     |
| **Frequency**     | 1 column          | Cheap, leak-free, often surprisingly good. | Loses identity if two categories share counts. |
| **Target (mean)** | 1 column          | Captures predictive signal directly.       | **High leakage risk** — fit on train only.|

### Pick a categorical column to demonstrate


In [16]:
import pandas as pd
candidates = [c for c in (STATIC_COLS + DYNAMIC_COLS)
              if c in df_feat.columns and not pd.api.types.is_numeric_dtype(df_feat[c])]
if not candidates:
    # Fall back to the first key column
    candidates = [KEY_COLS[0]]
example_col = candidates[0]
print(f'Encoding example on column: {example_col}')
print(f'Unique values: {df_feat[example_col].nunique()}')


Encoding example on column: warehouse
Unique values: 7


### Label encoding


In [17]:
df_lab, lab_maps = ft.features.label_encode(df_feat, [example_col])
print('First few mappings:', dict(list(lab_maps[example_col].items())[:5]))
df_lab[[example_col]].head()


First few mappings: {'Brno_1': 0, 'Budapest_1': 1, 'Frankfurt_1': 2, 'Munich_1': 3, 'Prague_1': 4}


,warehouse
0,1
1,1
2,1
3,1
4,1


### One-hot encoding


In [18]:
# Run on a small sample to keep the output tidy
df_oh = ft.features.one_hot_encode(df_feat.head(10), [example_col], drop_first=False)
new_cols = [c for c in df_oh.columns if c.startswith(f'{example_col}_')]
print(f'One-hot expanded to {len(new_cols)} columns:')
df_oh[new_cols]


One-hot expanded to 1 columns:


,warehouse_Budapest_1
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1


### Frequency encoding


In [19]:
df_freq = ft.features.frequency_encode(df_feat, [example_col])
df_freq[[example_col, f'{example_col}_freq']].drop_duplicates().head(10)


,warehouse,warehouse_freq
0,Budapest_1,803945
376,Frankfurt_1,258311
1111,Munich_1,367664
6203,Prague_1,937308
7619,Prague_3,939768
9035,Prague_2,929433
10451,Brno_1,789497


### Target encoding (with leakage warning)

Target encoding replaces a category with the **mean of the target** for
that category. It's powerful but dangerous: if you fit on the whole
dataset, you encode each row using its own target value 🤦. Always:

1. Fit `target_encode` on the **training** split only.
2. Use `apply_target_encoding` on validation and test splits.

The toolkit also smooths each category mean toward the global mean
(parameter `smoothing`) to dampen the influence of categories with few
observations.


In [20]:
# Use a placeholder train/test split for demo only.
# Notebook 7 covers proper splitting.
cut = int(0.8 * len(df_feat))
train_part = df_feat.iloc[:cut]
test_part  = df_feat.iloc[cut:]

train_te, encs = ft.features.target_encode(train_part, [example_col],
                                           target=TARGET_COL,
                                           smoothing=10.0)
test_te = ft.features.apply_target_encoding(test_part, encs)

print('Train rows with new column:')
display(train_te[[example_col, f'{example_col}_te']].drop_duplicates().head(10))
print('Global mean used as fallback for unseen categories:',
      f'{float(encs["__global_mean__"].iloc[0]):.3f}')


Train rows with new column:


,warehouse,warehouse_te
0,Budapest_1,76.929123
376,Frankfurt_1,35.112325
1111,Munich_1,69.489417
6203,Prague_1,112.305149
7619,Prague_3,59.705707
9035,Prague_2,57.840624
10451,Brno_1,116.124291


Global mean used as fallback for unseen categories: 80.233


## 6.7 Take-aways

| Family            | Recommendation                                                              |
|-------------------|-----------------------------------------------------------------------------|
| Date features     | Always include. Add `sin/cos` for cyclical attributes.                      |
| Lags              | Use a few canonical multiples of the seasonality you saw in Notebook 2.     |
| Rolling stats     | Always shift by ≥ 1 to avoid leakage.                                       |
| Numeric transforms| Try Yeo-Johnson when target has zeros/negatives; log(1+x) otherwise.        |
| Scaling           | Required for linear/NN models. Fit on **train only**, apply elsewhere.      |
| Encoding          | Tree models → label/frequency. Linear → one-hot. Target encoding → with care. |

You now have a feature matrix ready to be split. That's the next and
last preparation notebook.

Next: **Notebook 7 — Train / Validation / Test Split**.
